# Importing the required libraries

In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import optimizers, losses, activations, models
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten, MaxPooling2D, Dropout,LayerNormalization, TimeDistributed, BatchNormalization, LeakyReLU, Dropout, TimeDistributed, Input, Convolution1D, MaxPool1D, GlobalMaxPool1D
from tensorflow.keras.optimizers import Adam, Nadam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import StratifiedKFold
import time
from keras.callbacks import ModelCheckpoint, CSVLogger
from tensorflow.keras.callbacks import TensorBoard

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from tensorflow.keras import layers
from tensorflow.keras.layers import TimeDistributed, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt

import tensorflow as tf
# import os
# from tensorboard.plugins.hparams import api as hp



# Dataset


In [ ]:
mnist = tf.keras.datasets.mnist
(x, y),(x_test, y_test) = mnist.load_data()

#normalization
x, x_test = x/255.0, x_test/255.0

x_train, x_val, y_train, y_val = x[:50000], x[50000:], y[:50000], y[50000:] 

print("X train = ", x_train.shape)
print("y train = ", y_train.shape)
print("X validation = ", x_val.shape)
print("y validation = ", y_val.shape)
print("X test = ", x_test.shape)
print("y test = ", y_test.shape)



Visualizing MNIST dataset

In [ ]:
print("Shape of images: ", x_train.shape[1:])
print("Shape of labels: ", y_train.shape)

# fig, ax = plt.subplots(10, 10)
# k = 0
# for i in range(10):
#     for j in range(10):
#         ax[i][j].imshow(x_train[k], aspect = 'auto')
#         k += 1

# plt.show()

# Data pre-processing

In [ ]:
x_train = x_train.reshape(x_train.shape[0],28*28)
x_val = x_val.reshape(x_val.shape[0],28*28)
x_test = x_test.reshape(x_test.shape[0],28*28)

In [ ]:
print(x_train.shape, x_test.shape, x_val.shape)

In [ ]:
def plot_training_params(file_name, ann_model):
    plt.rcParams.update({'font.size': 8})

    plt.subplot(2,2,1)
    plt.plot(ann_model.history['acc'], label = 'training accuracy')
    plt.title('Accuracy')
    plt.xlabel('epochs')
    plt.ylabel('accuracy')
    plt.grid()

    plt.subplot(2,2,2)
    plt.plot(ann_model.history['val_acc'], label = 'validation accuracy')
    plt.title('Validation_Accuracy')
    plt.xlabel('epochs')
    plt.ylabel('val_accuracy')
    #plt.legend(legends)
    plt.grid()

    plt.subplot(2,2,3)
    plt.plot(ann_model.history['loss'], label = 'validation accuracy')
    plt.title('Loss')
    plt.xlabel('epochs')
    plt.ylabel('loss')
    #plt.legend(legends)
    plt.grid()

    plt.subplot(2,2,4)
    plt.plot(ann_model.history['val_loss'], label = 'validation accuracy')
    plt.title('Validation_Loss')
    plt.xlabel('epochs')
    plt.ylabel('val_loss')
    #plt.legend(legends)
    plt.grid()
    plt.tight_layout()
    
    plt.savefig(file_name + ".png",dpi=1200)
 


# ANN without regularization

In [ ]:
# Create model
def get_ANN_model():
    model=Sequential()
    model.add(Input(shape = (28*28,), name = 'Input_layer'))
    model.add(Dense(500, activation='relu', name='hidden_1'))
    model.add(Dense(500, activation='relu', name='hidden_2'))
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

# Model training

In [ ]:
tf.random.set_seed(42)
dropout_rate = [0.0, 0.2, 0.4]
model = get_ANN_model()
#es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 5)
ann_model_1 = model.fit(x=x_train,y=y_train,
                                batch_size=64,
                                epochs=1,
                                validation_data=(x_val, y_val),
                                shuffle=False,
                                verbose = 1)

plot_training_params("ANN_1", ann_model_1)

#cnn_model = model.fit(x=x_train,y=y_train,
#                                batch_size=64,
#                                epochs=250,
#                                validation_data=(x_val, y_val),
#                                shuffle=False,
#                                verbose = 1,
#                                callbacks=[es])

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test,batch_size = 64);
print("Test_accuracy: ", test_acc)
print("Test_loss: ", test_loss)

# Early stopping 

In [ ]:
tf.random.set_seed(42)
dropout_rate = [0.0, 0.2, 0.4]
model = get_ANN_model()
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 5)
ann_model_1 = model.fit(x=x_train,y=y_train,
                                batch_size=64,
                                epochs=250,
                                validation_data=(x_val, y_val),
                                shuffle=False,
                                callbacks = [es],
                                verbose = 1)

plot_training_params("ANN_1_early_stopping", ann_model_1)

#cnn_model = model.fit(x=x_train,y=y_train,
#                                batch_size=64,
#                                epochs=250,
#                                validation_data=(x_val, y_val),
#                                shuffle=False,
#                                verbose = 1,
#                                callbacks=[es])

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test,batch_size = 64);
print("Test_accuracy: ", test_acc)
print("Test_loss: ", test_loss)

# ANN model with L2_regularization

In [ ]:
# Create model
def get_ANN_L2_reg_model():
    model=Sequential()
    model.add(Input(shape = (28*28), name = 'Input_layer'))
    model.add(Dense(500, activation='relu', name='hidden_1', kernel_regularizer='l2'))
    #model.add(Dropout(dropout_rate))
    model.add(Dense(500, activation='relu', name='hidden_2', kernel_regularizer='l2'))
    #model.add(Dropout(0.2))
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [ ]:
tf.random.set_seed(42)

model = get_ANN_L2_reg_model()
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 5)
ann_model_1 = model.fit(x=x_train,y=y_train,
                                batch_size=64,
                                epochs=250,
                                validation_data=(x_val, y_val),
                                shuffle=False,
                                callbacks = [es],
                                verbose = 1)

plot_training_params("ANN_L2_reg", ann_model_1)

#

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test,batch_size = 64);
print("Test_accuracy: ", test_acc)
print("Test_loss: ", test_loss)

# ANN model with L2, regularization and dropout

In [ ]:
# Create model
def get_ANN_dropout_model():
    model=Sequential()
    model.add(Input(shape = (28*28), name = 'Input_layer'))
    model.add(Dense(500, activation='relu', name='hidden_1', kernel_regularizer='l2'))
    model.add(Dropout(0.4))
    model.add(Dense(500, activation='relu', name='hidden_2', kernel_regularizer='l2'))
    model.add(Dropout(0.2))
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [ ]:
tf.random.set_seed(42)

model = get_ANN_dropout_model()
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 5)
ann_model_1 = model.fit(x=x_train,y=y_train,
                                batch_size=64,
                                epochs=250,
                                validation_data=(x_val, y_val),
                                shuffle=False,
                                callbacks = [es],
                                verbose = 1)

plot_training_params("ANN_dropout_L2_", ann_model_1)

test_loss, test_acc = model.evaluate(x_test, y_test,batch_size = 64);
print("Test_accuracy: ", test_acc)
print("Test_loss: ", test_loss)